In [1]:
import numpy as np
import pandas as pd
import plotly.express as px

In [2]:
data_dir = "data/external"

## GMM shapes


In [3]:
def cmap(n):
    return px.colors.sample_colorscale("Turbo", [i / n for i in range(n)])

In [4]:
mean_shapes_df = np.load(f"{data_dir}/shape_cluser_means.npz")
mean_shapes_df = pd.DataFrame(
    mean_shapes_df["shapes"].T,
    columns=mean_shapes_df["labels"],
    index=mean_shapes_df["time"],
)
mean_shapes_df.index.name = "time [ps]"
fig = px.line(
    mean_shapes_df,
    y=mean_shapes_df.columns,
    title="GMM shapes",
)
for trace, c in zip(fig.data, cmap(len(fig.data))):
    trace.line.color = c
fig.show()

In [78]:
mean_lat_vecs_df = np.load(f"{data_dir}/shape_cluser_means.npz")
mean_lat_vecs_df = pd.DataFrame(
    mean_lat_vecs_df["latent_vecs"].T,
    columns=mean_lat_vecs_df["labels"],
)


def show_latent_vector_components_as_bars(lat_vecs_df):
    fig = px.bar(
        lat_vecs_df,
        x=lat_vecs_df.columns,
        y=lat_vecs_df.index,
        orientation="h",
        barmode="group",
        title="latent vector components of GMMs",
    )
    for trace, c in zip(fig.data, cmap(len(fig.data))):
        trace.marker.color = c
    fig.update_layout(
        height=30 * len(lat_vecs_df) + 100,
        bargap=0.6,
        bargroupgap=0.0,
    )
    fig.show()


def show_latent_vector_components_as_boxplot(lat_vecs_df):
    df = lat_vecs_df.T.melt(var_name="component", value_name="value")
    fig = px.box(
        df,
        x="component",
        y="value",
        title="distribution of latent vector components",
    )
    fig.update_layout(
        height=400,
    )
    fig.show()


def show_latent_vector_components_as_hists(lat_vecs_df, n_cols=8):
    df = lat_vecs_df.T.melt(var_name="component", value_name="value")
    n_components = len(df["component"].unique())
    fig = px.histogram(
        df,
        x="value",
        facet_col="component",
        facet_col_wrap=n_cols,
        title="distribution of latent vector components",
        nbins=30,
    )
    # rescale all x-axes to same range
    x_min = df["value"].min()
    x_max = df["value"].max()
    for i in range(n_components):
        fig.layout[f"xaxis{i+1}"].update(range=[x_min, x_max])
    # tight layout
    fig.update_layout(margin=dict(l=10, r=10, t=50, b=10))
    fig.update_layout(
        height=100 * ((n_components - 1) // n_cols + 1),
        width=100 * n_cols,
    )
    fig.show()


show_latent_vector_components_as_bars(mean_lat_vecs_df)
show_latent_vector_components_as_boxplot(mean_lat_vecs_df)
show_latent_vector_components_as_hists(mean_lat_vecs_df)

# Vary/Noisify Shapes


In [6]:
cluster_id_cmap = dict(
    zip(mean_lat_vecs_df.columns, cmap(len(mean_lat_vecs_df.columns)))
)

In [7]:
def augment_with_noise(df, n=1, noise_std=1):
    """
    Create n copies of each row in df and add Gaussian noise to numerical columns.

    Parameters:
    - df: pandas DataFrame
    - n: number of copies per row
    - noise_std: standard deviation of Gaussian noise relative to each value

    Returns:
    - augmented DataFrame
    """
    # identify numerical columns
    num_cols = df.select_dtypes(include=[np.number]).columns

    # repeat each row n times
    df_repeated = pd.DataFrame(np.repeat(df.values, n, axis=0), columns=df.columns)

    # add Gaussian noise to numerical columns
    for col in num_cols:
        noise = np.random.normal(loc=0, scale=noise_std, size=df_repeated.shape[0])
        df_repeated[col] = df_repeated[col].astype(float) + noise

    return df_repeated


# example
df = pd.DataFrame(
    {
        "A": [1, 2, 3],
        "B": [10, 20, 30],
        "C": ["x", "y", "z"],
    }
)

augment_with_noise(df, n=3, noise_std=0.5)

,A,B,C
0,1.340192,9.280997,x
1,0.887961,9.597202,x
2,1.664612,9.190243,x
3,1.745979,20.049873,y
4,1.677688,20.573924,y
5,1.436061,19.998391,y
6,3.177367,30.411412,z
7,2.487456,30.245315,z
8,3.574380,30.156636,z


In [ ]:
def _to_label_and_array(c):
    z = mean_lat_vecs_df[c]
    return z.name, z.to_numpy()


mean_lat_vec: dict = dict(map(_to_label_and_array, mean_lat_vecs_df.columns))

In [8]:
def augment_columns_with_noise(df, n=1, noise_std=1):
    """
    Create n copies of each column in df and add Gaussian noise to numerical columns.

    Parameters:
    - df: pandas DataFrame
    - n: number of copies per column
    - noise_std: standard deviation of Gaussian noise relative to each value

    Returns:
    - augmented DataFrame with more columns
    """
    num_cols = df.select_dtypes(include=[np.number]).columns
    augmented_data = {}

    for col in df.columns:
        for i in range(n):
            new_col_name = f"{col}-{i+1}"
            if col in num_cols:
                # add Gaussian noise
                augmented_data[new_col_name] = df[col].astype(float) + np.random.normal(
                    0, noise_std, size=len(df)
                )
            else:
                # just replicate non-numeric column
                augmented_data[new_col_name] = df[col]

    return pd.DataFrame(augmented_data)


# example
df = pd.DataFrame(
    {
        "A": [1, 2, 3],
        "B": [10, 20, 30],
        "C": ["x", "y", "z"],
    }
)

augment_columns_with_noise(df, n=3, noise_std=0.5)

,A-1,A-2,A-3,B-1,B-2,B-3,C-1,C-2,C-3
0,1.602006,1.266370,-0.032592,9.980972,9.348655,10.430123,x,x,x
1,1.640931,2.858811,1.581895,20.051280,19.564674,20.950896,y,y,y
2,2.698077,3.146060,2.925020,29.282320,30.049461,29.741870,z,z,z


In [ ]:
def noisify_lattent_vectors(lat_vec_df, n_variations: int = 1, noise_std: float = 1):

    return (
        augment_columns_with_noise(
            lat_vec_df,
            n=n_variations,
            noise_std=noise_std,
        ),
        [c for c in lat_vec_df.columns for _ in range(n_variations)],
    )

In [87]:
for std in [0.001, 0.01, 0.1, 1, 10]:
    print(f"noise_std: {std}")
    mean_lat_vecs_with_noise_df, mean_lat_vecs_with_noise_df_cluster_ids = (
        noisify_lattent_vectors(mean_lat_vecs_df, n_variations=100, noise_std=std)
    )
    show_latent_vector_components_as_boxplot(mean_lat_vecs_with_noise_df)

noise_std: 0.001


noise_std: 0.01


noise_std: 0.1


noise_std: 1


noise_std: 10
